In [ ]:
import os
os.chdir(r'F:\\Users\\Project\\MWM\\decoding_python') # change this address to your directory 
data_dir='F:/Users/Project/MWM/data/' # change this address to your directory 
os.getcwd()

In [ ]:

import numpy as np
from scipy import io
import sys
import pandas as pd

import os.path
from os import path
import pickle
###Import functions for binning data for preprocessing###
from Neural_Decoding_NEW.preprocessing_funcs import bin_spikes
from Neural_Decoding_NEW.preprocessing_funcs import bin_spikes2
from Neural_Decoding_NEW.preprocessing_funcs import bin_output
from Neural_Decoding_NEW.preprocessing_funcs import bin_output_pos
from Neural_Decoding_NEW.preprocessing_funcs import bin_spikes_hist

## Original Data

In [ ]:


all_anm=[
'Animal17H'
]

all_cells=0
random_trials=1         
#all_anm=['Animal24H']
for N_repetition in range (1,11):
    print(N_repetition)
    for anm in all_anm:
        
       # for flag_pos_HD_G in range (0,4):
        for flag_pos_HD_G in range (0,1):
            for DDAY in range (1,2):
                for DDAY2 in range (6,7):
                    day=str(DDAY)
                    day2=str(DDAY2)
                    sp_day=1
                    Animal_number=anm
                   # flag_pos_HD_G=0 #0: pos  
                    if flag_pos_HD_G==0:
                        folder=data_dir+'result/decoding/sample_data_RNN/concatenate_all_anm_same/'+Animal_number+'/'
                        data=io.loadmat(folder+'MWM_pos_day_'+day+'_day_'+day2+'_'+str(sp_day)+'_all_cells_'+str(all_cells)+'_random_trials_'+str(random_trials)+'_N_repetition'+str(N_repetition)+'.mat')
                       
                        
                        
                        
    
                    spike_times=data['spike_times'] #Load spike times of all neurons
                    pos=data['pos'] #Load x and y positions
                    pos_times=data['pos_times'][0] #Load times at which positions were recorded
                    trace=data['trace']
                    mfr=data['deconv_trace']
    
                    ################################### User Inputs
    
                    dt=.1 #Size of time bins (in seconds)
                    t_start=pos_times[0] #Time to start extracting data - here the first time position was recorded
                    t_end=pos_times[-1]#5608 #pos_times[-1] #Time to finish extracting data - when looking through the dataset, the final position was recorded around t=5609, but the final spikes were recorded around t=5608
                    downsample_factor=1 #Downsampling of output (to make binning go faster). 1 means no downsampling.
    
                    ################################### Put data in binned format
                    #When loading the Matlab cell "spike_times", Python puts it in a format with an extra unnecessary dimension
                    #First, we will put spike_times in a cleaner format: an array of arrays
                    spike_times=np.squeeze(spike_times)
                    trace=np.squeeze(trace)
                    mfr=np.squeeze(mfr)
                    for i in range(spike_times.shape[0]):
                        spike_times[i]=np.squeeze(spike_times[i])
                        trace[i]=np.squeeze(trace[i])
                        mfr[i]=np.squeeze(mfr[i])
    
                    ################################### Put data in binned format
                    data2 = {'Time':  pos_times,
                            }
    
                    df = pd.DataFrame (data2)
                    for i in range(1,len(trace)+1):
                        df[i] = trace[i-1] 
                    df2 = pd.DataFrame (data2)
                    for i in range(1,len(mfr)+1):
                        df2[i] = mfr[i-1] 
    
    
                    ###Preprocessing to put spikes and output in bins###
    
                    #Bin neural data using "bin_spikes" function
                    neural_data=bin_spikes(spike_times,dt,t_start,t_end)
                    #neural_data_df=bin_spikes2(spike_times,df,dt,t_start,t_end)
                    neural_data_df=bin_spikes_hist(spike_times,df,dt,t_start,t_end)
                    neural_data_df2=bin_spikes_hist(spike_times,df2,dt,t_start,t_end)
                    if flag_pos_HD_G==0:
                        pos_binned=bin_output_pos(pos,pos_times,dt,t_start,t_end,downsample_factor)
                    if flag_pos_HD_G==1:
                        pos_binned=bin_output(pos,pos_times,dt,t_start,t_end,downsample_factor)
                        
                    if flag_pos_HD_G==2:
                        pos_binned=bin_output_distance(pos,pos_times,dt,t_start,t_end,downsample_factor)               
                    if flag_pos_HD_G==3:
                        pos_binned=bin_output_distance(pos,pos_times,dt,t_start,t_end,downsample_factor)  
                        
                    if flag_pos_HD_G==4:
                        pos_binned=bin_output(pos,pos_times,dt,t_start,t_end,downsample_factor)
                                        
                                  
                                        
                    if flag_pos_HD_G==5:
                        pos_binned=bin_output(pos,pos_times,dt,t_start,t_end,downsample_factor)
                    if flag_pos_HD_G==6:
                        pos_binned=bin_output_all(pos,pos_times,dt,t_start,t_end,downsample_factor)
                    
                    if flag_pos_HD_G==7:
                        pos_binned=bin_output_distance(pos,pos_times,dt,t_start,t_end,downsample_factor)               
    
    
    
             
                    edges=np.arange(t_start,t_end,dt) #Get edges of time bins
                    num_bins=edges.shape[0]-1 #Number of bins
                    output_dim=pos.shape[1] #Number of output features
                    outputs_binned=np.empty([num_bins,output_dim]) #Ini
    
    
    
                    ################################### Save Data
    
                    #data_folder='/Users/minashahi/OneDrive/Documents/UCLA/Project/MWM/sample_data_RNN/concatenate/result_HD/' #FOLDER YOU WANT TO SAVE THE DATA TO
                    if flag_pos_HD_G==0:
                        data_folder=data_dir+'result/decoding/sample_data_RNN/concatenate_all_anm_same/'+Animal_number+'/result_pos/' #FOLDER YOU WANT TO SAVE THE DATA TO
                        if not path.exists(data_folder):
                            os.mkdir(data_folder) 
                        with open(data_folder+'example_MWM_pos_day_'+day+'_day_'+day2+'_'+str(sp_day)+'_all_cells_'+str(all_cells)+'_random_trials_'+str(random_trials)+'_N_repetition'+str(N_repetition)+'.pickle','wb') as f:
                            pickle.dump([neural_data,pos_binned,neural_data_df,neural_data_df2],f)
                   
    
    


## Shuffled data

In [ ]:

all_anm=[
'Animal17H'

]


all_cells=0
sp_day=6
#all_anm=['Animal26H']
for N_repetition in range (1,11):
    for shuff in range(1,51):
        print(shuff)
        for anm in all_anm:
            for flag_pos_HD_G in range (15,16):

                for DDAY in range (1,2):
                    day=str(DDAY)
                    for DDAY2 in range (6,7):
                        day=str(DDAY)
                        day2=str(DDAY2)
                        Animal_number=anm
                       
 
                        if flag_pos_HD_G==15:
                            folder=data_dir+'result/decoding/sample_data_RNN/concatenate_all_anm_same/'+Animal_number+'/'
                            data=io.loadmat(folder+'MWM_shift_spike_pos_day_'+day+'_day_'+day2+'_shuff_'+str(shuff)+'_'+str(sp_day)+'_all_cells_'+str(all_cells)+'_random_trials_'+str(random_trials)+'_N_repetition'+str(N_repetition)+'.mat')
    
     
                     
                        spike_times=data['spike_times'] #Load spike times of all neurons
                        pos=data['pos'] #Load x and y positions
                        pos_times=data['pos_times'][0] #Load times at which positions were recorded
                        trace=data['trace']
                        mfr=data['deconv_trace']
    
                        ################################### User Inputs
    
                        dt=.1 #Size of time bins (in seconds)
                        t_start=pos_times[0] #Time to start extracting data - here the first time position was recorded
                        t_end=pos_times[-1]#5608 #pos_times[-1] #Time to finish extracting data - when looking through the dataset, the final position was recorded around t=5609, but the final spikes were recorded around t=5608
                        downsample_factor=1 #Downsampling of output (to make binning go faster). 1 means no downsampling.
    
                        ################################### Put data in binned format
                        #When loading the Matlab cell "spike_times", Python puts it in a format with an extra unnecessary dimension
                        #First, we will put spike_times in a cleaner format: an array of arrays
                        spike_times=np.squeeze(spike_times)
                        trace=np.squeeze(trace)
                        mfr=np.squeeze(mfr)
                        for i in range(spike_times.shape[0]):
                            spike_times[i]=np.squeeze(spike_times[i])
                            trace[i]=np.squeeze(trace[i])
                            mfr[i]=np.squeeze(mfr[i])
    
                        ################################### Put data in binned format
                        data2 = {'Time':  pos_times,
                                }
    
                        df = pd.DataFrame (data2)
                        for i in range(1,len(trace)+1):
                            df[i] = trace[i-1] 
                        df2 = pd.DataFrame (data2)
                        for i in range(1,len(mfr)+1):
                            df2[i] = mfr[i-1] 
    
    
                        ###Preprocessing to put spikes and output in bins###
    
                        #Bin neural data using "bin_spikes" function
                        neural_data=bin_spikes(spike_times,dt,t_start,t_end)
                        #neural_data_df=bin_spikes2(spike_times,df,dt,t_start,t_end)
                        neural_data_df=bin_spikes_hist(spike_times,df,dt,t_start,t_end)
                        neural_data_df2=bin_spikes_hist(spike_times,df2,dt,t_start,t_end)
    
    
                        
                        if flag_pos_HD_G==15:
                            pos_binned=bin_output_pos(pos,pos_times,dt,t_start,t_end,downsample_factor)
                      
                        edges=np.arange(t_start,t_end,dt) #Get edges of time bins
                        num_bins=edges.shape[0]-1 #Number of bins
                        output_dim=pos.shape[1] #Number of output features
                        outputs_binned=np.empty([num_bins,output_dim]) #Ini
    
    
    
                        ################################### Save Data
    

                                
                        if flag_pos_HD_G==15:
                            data_folder=data_dir+'result/decoding/sample_data_RNN/concatenate_all_anm_same/'+Animal_number+'/result_shift_pos/' #FOLDER YOU WANT TO SAVE THE DATA TO
                            if not path.exists(data_folder):
                                os.mkdir(data_folder)
                                print('yes')
                            with open(data_folder+'example_MWM_shift_pos_day_'+day+'_day_'+day2+'_shuff_'+str(shuff)+'_'+str(sp_day)+'_all_cells_'+str(all_cells)+'_random_trials_'+str(random_trials)+'_N_repetition'+str(N_repetition)+'.pickle','wb') as f:
                                pickle.dump([neural_data,pos_binned,neural_data_df,neural_data_df2],f)
                      
                       


In [ ]:
sp_day=1
all_cells=0
random_trials=1
k_fold=5

## Position

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from pylab import *
%matplotlib inline
from scipy import io
from scipy import stats
import pickle
import sys
import math
import astropy
import itertools

import os.path
from os import path
from scipy.io import savemat

#Import function to get the covariate matrix that includes spike history from previous bins
from Neural_Decoding.preprocessing_funcs import get_spikes_with_history

#Import metrics
from Neural_Decoding.metrics import get_R2
from Neural_Decoding.metrics import get_rho

#Import decoder functions
from Neural_Decoding.decoders import WienerCascadeDecoder
from Neural_Decoding.decoders import WienerFilterDecoder
from Neural_Decoding.decoders import DenseNNDecoder
from Neural_Decoding.decoders import SimpleRNNDecoder
from Neural_Decoding.decoders import GRUDecoder
from Neural_Decoding.decoders import LSTMDecoder
from Neural_Decoding.decoders import XGBoostDecoder
from Neural_Decoding.decoders import SVRDecoder

bin_size=[10]
units_N=[1500]
dropout_N=[.2]
num_epochs_N=[15]





Animal_number_all=[
'Animal17H'

]



#Animal_number_all=['Animal13H']
str_pos_HD_G_all=['pos']
fold=6
#str_pos_HD_G_all=['distance']
DAY2=[6, 5 ,1, 3, 4 ,2]
# folder='/home/jglaser/Data/DecData/' 
# folder='/Users/jig289/Dropbox/Public/Decoding_Data/'
for N_repetition in range (1,11):
    for str_pos_HD_G in str_pos_HD_G_all:
        for Animal_number in Animal_number_all:
            print(Animal_number)
            folder=data_dir+'result/decoding/sample_data_RNN/concatenate_all_anm_same/'+Animal_number+'/result_'+str_pos_HD_G+'/' #ENTER THE FOLDER THAT YOUR DATA IS IN
            zip_list = list(itertools.product(bin_size,dropout_N,num_epochs_N))
            for count,zip_N in enumerate(zip_list):
                bins_before=zip_N[0] #4How many bins of neural data prior to the output are used for decoding
                bins_current=1 #5Whether to use concurrent time bin of neural data
                bins_after=zip_N[0] #5How many bins of neural data after the output are used for decoding
                N_day=6
                #Animal_number='Animal13H'
                #str_pos_HD_G='shift_pos'
                #folder='/Users/minashahi/OneDrive/Documents/UCLA/Project/MWM/sample_data_RNN/concatenate/mayank/result_HD/' #ENTER THE FOLDER THAT YOUR DATA IS IN
                folder=data_dir+'result/decoding/sample_data_RNN/concatenate_all_anm_same/'+Animal_number+'/result_'+str_pos_HD_G+'/' #ENTER THE FOLDER THAT YOUR DATA IS IN
    
                # folder='/home/jglaser/Data/DecData/' 
                            # folder='/Users/jig289/Dropbox/Public/Decoding_Data/'
                for day in range(1,2):
                   # print(day)
                    for day2 in range(6,7):
                        day2=str(day2)
                        day=str(day)
    
                        #with open(folder+'example_data_hc.pickle','rb') as f:
                        with open(folder+'example_MWM_'+str_pos_HD_G+'_day_'+day+'_day_'+day2+'_'+str(sp_day)+'_all_cells_'+str(all_cells)+'_random_trials_'+str(random_trials)+'_N_repetition'+str(N_repetition)+'.pickle','rb') as f:
                        #     neural_data,pos_binned=pickle.load(f,encoding='latin1') #If using python 3
                            neural_data,pos_binned,neural_data_df,neural_data_df2=pickle.load(f) #If using python 2
                      
                        #Remove neurons with too few spikes in HC dataset
                        nd_sum=np.nansum(neural_data,axis=0) #Total number of spikes of each neuron
                        rmv_nrn=np.where(nd_sum<80) #Find neurons who have less than 100 spikes total
                        neural_data=np.delete(neural_data,rmv_nrn,1) #Remove those neurons
                        neural_data_df=np.delete(neural_data_df,rmv_nrn,1)
                        # Format for recurrent neural networks (SimpleRNN, GRU, LSTM)
                        # Function to get the covariate matrix that includes spike history from previous bins
                        #X=get_spikes_with_history(neural_data,bins_before,bins_after,bins_current)
                        X_df=get_spikes_with_history(neural_data_df,bins_before,bins_after,bins_current)
                        X=get_spikes_with_history(neural_data_df,bins_before,bins_after,bins_current)
    
                        # Format for Wiener Filter, Wiener Cascade, XGBoost, and Dense Neural Network
                        #Put in "flat" format, so each "neuron / time" is a single feature
                        #X_flat=X.reshape(X.shape[0],(X.shape[1]*X.shape[2]))
                        X_flat_df=X_df.reshape(X_df.shape[0],(X_df.shape[1]*X_df.shape[2]))
                        #X_flat=X.reshape(X.shape[0],(X.shape[1]*X.shape[2]))
                        X_flat=X_df.reshape(X_df.shape[0],(X_df.shape[1]*X_df.shape[2]))
    
                        y=pos_binned
    
    
                        #Remove time bins with no output (y value)
                        #rmv_time=np.where(np.isnan(y[:,0]) | np.isnan(y[:,1]) | np.isnan(y[:,2])) #Find time bins with no output
                        AA=np.isnan(X_flat_df[:,1])
                        mean_N_spikes=np.mean(X_flat,1)
                        for indd in range(len(X_flat_df[0])):
                            AA=AA|np.isnan(X_flat_df[:,indd])
                        rmv_time=np.where(np.isnan(y[:,0]) | np.isnan(y[:,1]) | AA ) #Find time bins with no output
                        ####rmv_time=np.where(np.isnan(y[:,0])) #Find time bins with no output
                        X=np.delete(X,rmv_time,0) #Remove those time bins from X
                        X_flat=np.delete(X_flat,rmv_time,0) #Remove those time bins from X_flat
                        y=np.delete(y,rmv_time,0) #Remove those time bins from y
                        X_df=np.delete(X_df,rmv_time,0)
                        X_flat_df=np.delete(X_flat_df,rmv_time,0) #Remove those time bins from X

    
                        training_range=[0, 0.8]
                        valid_range=[0.8,1]
                        testing_range=[0.85, 1]
    
    
                        num_examples=X.shape[0]
                        X_flat_train_all=[]
                        X_flat_valid_all=[]
                        y_train_all=[]
                        y_valid_all=[]
    

    
                        training_set=np.arange(np.int_(np.round(training_range[0]*num_examples))+bins_before,np.int_(np.round(training_range[1]*num_examples))-bins_after)
                        testing_set=np.arange(np.int_(np.round(testing_range[0]*num_examples))+bins_before,np.int_(np.round(testing_range[1]*num_examples))-bins_after)
                        valid_set=np.arange(np.int_(np.round(valid_range[0]*num_examples))+bins_before,np.int_(np.round(valid_range[1]*num_examples))-bins_after)
                        ######all_set=np.arange(np.int(np.round(0*num_examples))+bins_before,np.int(np.round(1*num_examples))-bins_after)
                        #l3 = [x for x in l1 if x not in l2]
                        #Get training data
    
                        X_train=X[training_set,:,:]
                        X_flat_train=X_flat[training_set,:]
                        y_train=y[training_set,:]
    
                        #Get testing data
                        X_test=X[testing_set,:,:]
                        X_flat_test=X_flat[testing_set,:]
                        y_test=y[testing_set,:]
    
                        #Get validation data
                        X_valid=X[valid_set,:,:]
                        X_flat_valid=X_flat[valid_set,:]
                        y_valid=y[valid_set,:]
                        X_flat_df_valid=X_flat_df[valid_set,:]
                        
                        
    
    
                        units_N2=[X_flat_train.shape[1],X_flat_train.shape[1]]
                        model_dnn=DenseNNDecoder(units=units_N2,dropout=zip_N[1],num_epochs=zip_N[2])
    
    
                        #Fit model0
                        model_dnn.fit(X_flat_train,y_train)
    
                        #Get predictions
                        y_valid_predicted_dnn=model_dnn.predict(X_flat_valid)
                        #y_test_predicted_dnn=model_dnn.predict(X_flat_test)
    
                        #Get metric of fit
                        R2s_dnn=get_R2(y_valid,y_valid_predicted_dnn)
                        y_valid_predicted_dnn_train=model_dnn.predict(X_flat_train)
                        R2s_dnn_train=get_R2(y_train,y_valid_predicted_dnn_train)
                        print('R2s_train:', R2s_dnn_train)
                        print('R2s_sin:', R2s_dnn)
    

    
    
                        #data_folder='/Users/minashahi/OneDrive/Documents/UCLA/Project/MWM/sample_data_RNN/concatenate/result_HD/' #FOLDER YOU WANT TO SAVE THE DATA TO
                        data_folder=data_dir+'result/decoding/sample_data_RNN/concatenate_all_anm_same/'+Animal_number+'/result_'+str_pos_HD_G+'/' #FOLDER YOU WANT TO SAVE THE DATA TO
                        if not path.exists(data_folder):
                            os.mkdir(data_folder) 

                        with open(data_folder+'MWM_'+str_pos_HD_G+'_pred_'+'day_'+day+'_day_'+day2+'_'+str(sp_day)+'_all_cells_'+str(all_cells)+'_random_trials_'+str(random_trials)+'_fold_'+str(fold)+'_N_repetition'+str(N_repetition)+'.pickle','wb') as f:
                            pickle.dump([y_valid,y_valid_predicted_dnn,y_train,y_valid_predicted_dnn_train],f)
                            savemat(data_dir+'result/decoding/sample_data_RNN/concatenate_all_anm_same/'+Animal_number+'/result_'+str_pos_HD_G+'/'+str_pos_HD_G+'_day_'+day+'_day_'+day2+'_'+str(sp_day)+'_all_cells_'+str(all_cells)+'_random_trials_'+str(random_trials)+'_fold_'+str(fold)+'_N_repetition'+str(N_repetition)+'.mat', mdict={'y_valid': np.array(y_valid),'y_valid_predicted_dnn':np.array(y_valid_predicted_dnn),'y_train': np.array(y_train), 'y_valid_predicted_dnn_train': np.array(y_valid_predicted_dnn_train),'bins_before':np.array(bins_before),'bins_after':np.array(bins_after),'bins_current':np.array(bins_current),'dropout_N':np.array(dropout_N),'num_epochs_N':np.array(num_epochs_N),'units_N2':np.array(units_N2),})


## Shuffled Position 

In [ ]:

bin_size=[10]
units_N=[1500]
dropout_N=[.2]
num_epochs_N=[15]



#Animal_number_all=['Animal13H','Animal17H','Animal20H','Animal21H','Animal22H','Animal23H']



Animal_number_all=[
'Animal17H'

]

str_pos_HD_G_all=['shift_pos']

DAY2=[6, 5 ,1, 3, 4 ,2]
# folder='/home/jglaser/Data/DecData/' 
# folder='/Users/jig289/Dropbox/Public/Decoding_Data/'
for N_repetition in range (1,11):
    for shuff in range(1,51):
        print(shuff)
        for str_pos_HD_G in str_pos_HD_G_all:
            for Animal_number in Animal_number_all:
                #print(Animal_number)
                #print(str_pos_HD_G)
                folder=data_dir+'result/decoding/sample_data_RNN/concatenate_all_anm_same/'+Animal_number+'/result_'+str_pos_HD_G+'/' #ENTER THE FOLDER THAT YOUR DATA IS IN
                zip_list = list(itertools.product(bin_size,dropout_N,num_epochs_N))
                for count,zip_N in enumerate(zip_list):
                    bins_before=zip_N[0] #4How many bins of neural data prior to the output are used for decoding
                    bins_current=1 #5Whether to use concurrent time bin of neural data
                    bins_after=zip_N[0] #5How many bins of neural data after the output are used for decoding
                    N_day=6
                    #Animal_number='Animal13H'
                    #str_pos_HD_G='shift_pos'
                    #folder='/Users/minashahi/OneDrive/Documents/UCLA/Project/MWM/sample_data_RNN/concatenate/mayank/result_HD/' #ENTER THE FOLDER THAT YOUR DATA IS IN
                    folder=data_dir+'result/decoding/sample_data_RNN/concatenate_all_anm_same/'+Animal_number+'/result_'+str_pos_HD_G+'/' #ENTER THE FOLDER THAT YOUR DATA IS IN
    
                    # folder='/home/jglaser/Data/DecData/' 
                                # folder='/Users/jig289/Dropbox/Public/Decoding_Data/'
                    for day in range(1,2):
                        
                        #day2=str(day+1)
                        for day2 in range(6,7):
                            day2=str(day2)
                            day=str(day)
    
                            #with open(folder+'example_data_hc.pickle','rb') as f:
                            with open(folder+'example_MWM_'+str_pos_HD_G+'_day_'+day+'_day_'+day2+'_shuff_'+str(shuff)+'_'+str(sp_day)+'_all_cells_'+str(all_cells)+'_random_trials_'+str(random_trials)+'_N_repetition'+str(N_repetition)+'.pickle','rb') as f:
                            #     neural_data,pos_binned=pickle.load(f,encoding='latin1') #If using python 3
                                neural_data,pos_binned,neural_data_df,neural_data_df2=pickle.load(f) #If using python 2
                          
    
                            #Remove neurons with too few spikes in HC dataset
                            nd_sum=np.nansum(neural_data,axis=0) #Total number of spikes of each neuron
                            rmv_nrn=np.where(nd_sum<80) #Find neurons who have less than 100 spikes total
                            neural_data=np.delete(neural_data,rmv_nrn,1) #Remove those neurons
                            neural_data_df=np.delete(neural_data_df,rmv_nrn,1)
                            # Format for recurrent neural networks (SimpleRNN, GRU, LSTM)
                            # Function to get the covariate matrix that includes spike history from previous bins
                            #X=get_spikes_with_history(neural_data,bins_before,bins_after,bins_current)
                            X_df=get_spikes_with_history(neural_data_df,bins_before,bins_after,bins_current)
                            X=get_spikes_with_history(neural_data_df,bins_before,bins_after,bins_current)
    
                            # Format for Wiener Filter, Wiener Cascade, XGBoost, and Dense Neural Network
                            #Put in "flat" format, so each "neuron / time" is a single feature
                            #X_flat=X.reshape(X.shape[0],(X.shape[1]*X.shape[2]))
                            X_flat_df=X_df.reshape(X_df.shape[0],(X_df.shape[1]*X_df.shape[2]))
                            #X_flat=X.reshape(X.shape[0],(X.shape[1]*X.shape[2]))
                            X_flat=X_df.reshape(X_df.shape[0],(X_df.shape[1]*X_df.shape[2]))
    
                            y=pos_binned
    
    
                            #Remove time bins with no output (y value)
                            #rmv_time=np.where(np.isnan(y[:,0]) | np.isnan(y[:,1]) | np.isnan(y[:,2])) #Find time bins with no output
                            AA=np.isnan(X_flat_df[:,1])
                            mean_N_spikes=np.mean(X_flat,1)
                            for indd in range(len(X_flat_df[0])):
                                AA=AA|np.isnan(X_flat_df[:,indd])
                            rmv_time=np.where(np.isnan(y[:,0]) | np.isnan(y[:,1]) | AA ) #Find time bins with no output
                            ####rmv_time=np.where(np.isnan(y[:,0])) #Find time bins with no output
                            X=np.delete(X,rmv_time,0) #Remove those time bins from X
                            X_flat=np.delete(X_flat,rmv_time,0) #Remove those time bins from X_flat
                            y=np.delete(y,rmv_time,0) #Remove those time bins from y
                            X_df=np.delete(X_df,rmv_time,0)
                            X_flat_df=np.delete(X_flat_df,rmv_time,0) #Remove those time bins from X
                           
                            training_range=[0, 0.8]
                            valid_range=[0.8,1]
                            testing_range=[0.85, 1]
    
    
                            num_examples=X.shape[0]
    

                            training_set=np.arange(np.int_(np.round(training_range[0]*num_examples))+bins_before,np.int_(np.round(training_range[1]*num_examples))-bins_after)
                            testing_set=np.arange(np.int_(np.round(testing_range[0]*num_examples))+bins_before,np.int_(np.round(testing_range[1]*num_examples))-bins_after)
                            valid_set=np.arange(np.int_(np.round(valid_range[0]*num_examples))+bins_before,np.int_(np.round(valid_range[1]*num_examples))-bins_after)
    
                            #Get training data
                            X_train=X[training_set,:,:]
                            X_flat_train=X_flat[training_set,:]
                            y_train=y[training_set,:]
    
                            #Get testing data
                            X_test=X[testing_set,:,:]
                            X_flat_test=X_flat[testing_set,:]
                            y_test=y[testing_set,:]
    
                            #Get validation data
                            X_valid=X[valid_set,:,:]
                            X_flat_valid=X_flat[valid_set,:]
                            y_valid=y[valid_set,:]
                            X_flat_df_valid=X_flat_df[valid_set,:]
    
                            import matplotlib.pyplot as plt
                            plt.hist(y_valid)
    
                            X_flat_train.shape[1]
    
                            XX=X_train[1:3,1:13,1:4]
                            xx=[]
    
    
    
    
    
    
                            #Z-score "X" inputs. 
                            X_train_mean=np.nanmean(X_train,axis=0)
                            X_train_std=np.nanstd(X_train,axis=0)

    
                            #Z-score "X_flat" inputs. 
                            X_flat_train_mean=np.nanmean(X_flat_train,axis=0)
                            X_flat_train_std=np.nanstd(X_flat_train,axis=0)

    
                            #Zero-center outputs
                            y_train_mean=np.mean(y_train,axis=0)


    
                            units_N2=[X_flat_train.shape[1],X_flat_train.shape[1]]
                            model_dnn=DenseNNDecoder(units=units_N2,dropout=zip_N[1],num_epochs=zip_N[2])
    
    
                            #Fit model0
                            model_dnn.fit(X_flat_train,y_train)
    
                            #Get predictions
                            y_valid_predicted_dnn=model_dnn.predict(X_flat_valid)
                            y_test_predicted_dnn=model_dnn.predict(X_flat_test)
    
                            #Get metric of fit
                            R2s_dnn=get_R2(y_valid,y_valid_predicted_dnn)
                            y_valid_predicted_dnn_train=model_dnn.predict(X_flat_train)
                            R2s_dnn_train=get_R2(y_train,y_valid_predicted_dnn_train)
                            print('R2s_train:', R2s_dnn_train)
                            print('R2s_sin:', R2s_dnn)
    
                           #y_train
                           # y_valid_predicted_dnn_train
    
    
                            #data_folder='/Users/minashahi/OneDrive/Documents/UCLA/Project/MWM/sample_data_RNN/concatenate/result_HD/' #FOLDER YOU WANT TO SAVE THE DATA TO
                            data_folder=data_dir+'result/decoding/sample_data_RNN/concatenate_all_anm_same/'+Animal_number+'/result_'+str_pos_HD_G+'/' #FOLDER YOU WANT TO SAVE THE DATA TO
                            if not path.exists(data_folder):
                                os.mkdir(data_folder) 
                            # data_folder='/home/jglaser/Data/DecData/' 
    
                            #with open(data_folder+'example_data_hc.pickle','wb') as f:
                             #   pickle.dump([neural_data,pos_binned],f)
                            with open(data_folder+'MWM_'+str_pos_HD_G+'_pred_day_'+day+'_day_'+day2+'_shuff_'+str(shuff)+'_'+str(sp_day)+'_all_cells_'+str(all_cells)+'_random_trials_'+str(random_trials)+'_N_repetition'+str(N_repetition)+'.pickle','wb') as f:
                                pickle.dump([y_valid,y_valid_predicted_dnn,y_train,y_valid_predicted_dnn_train],f)
                                savemat(data_dir+'result/decoding/sample_data_RNN/concatenate_all_anm_same/'+Animal_number+'/result_'+str_pos_HD_G+'/'+'shift_spike_pos'+'_day_'+day+'_day_'+day2+'_shuff_'+str(shuff)+'_'+str(sp_day)+'_all_cells_'+str(all_cells)+'_random_trials_'+str(random_trials)+'_N_repetition'+str(N_repetition)+'.mat', mdict={'y_valid': np.array(y_valid),'y_valid_predicted_dnn':np.array(y_valid_predicted_dnn),'y_train': np.array(y_train), 'y_valid_predicted_dnn_train': np.array(y_valid_predicted_dnn_train),'bins_before':np.array(bins_before),'bins_after':np.array(bins_after),'bins_current':np.array(bins_current),'dropout_N':np.array(dropout_N),'num_epochs_N':np.array(num_epochs_N),'units_N2':np.array(units_N2),})
    
    
